# Kashmiri Whisper Large-V3-Turbo: Ultra-Accuracy Stage-2 Fine-Tuning

This notebook performs Stage-2 precision fine-tuning on **OpenAI Whisper Large-V3-Turbo (809M)** for Kashmiri Speech-to-Text:
- **Warm-Start**: Initializes weights from `Sameer0313/kashmiri-whisper-lora-v2` (CER: 22.34%).
- **Licensed Public Speech**: Audits and imports OpenSLR-122 plus gated AI4Bharat Kashmiri speech after access approval.
- **Cross-Attention & Deep LoRA**: Upgrades to Rank=128, Alpha=256 across all attention and MLP layers.
- **SpecAugment**: Frequency and Time masking on Mel spectrograms for noise resilience.
- **Orthographic Normalization**: Unifies Persian-Arabic Kashmiri character variants to minimize character error rate.
- **Discourse Prompting**: Injects authentic Kashmiri discourse particles and syntactic cues.

In [ ]:
# 1. Install Dependencies & Clone Cipher Repo
!pip install -q transformers peft evaluate soundfile librosa jiwer huggingface_hub accelerate datasets pyarrow
!git clone https://github.com/sameerahmad86232/cipher.git /kaggle/working/cipher

In [ ]:
# 2. Prepare licensed public Kashmiri speech
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    print("HF_TOKEN is not configured; gated AI4Bharat sources will be skipped.")
include_gated = "--include-gated" if os.getenv("HF_TOKEN") else ""
!python3 /kaggle/working/cipher/scripts/prepare-public-kashmiri-asr.py --output /kaggle/working/public-kashmiri-asr {include_gated}

# 3. Launch Ultra-Accuracy Fine-Tuning

os.environ["NUM_EPOCHS"] = "12"
os.environ["BATCH_SIZE"] = "4"
os.environ["GRAD_ACCUM"] = "4"
os.environ["LEARNING_RATE"] = "7.5e-5"
os.environ["STAGE1_ADAPTER"] = "Sameer0313/kashmiri-whisper-lora-v2"
os.environ["EXTRA_AUDIO_ROOTS"] = "/kaggle/working/public-kashmiri-asr"
os.environ["OUTPUT_DIR"] = "/kaggle/working/kashmiri-whisper-ultra-adapter"
os.environ["CHECKPOINTS_DIR"] = "/kaggle/working/whisper-ultra-checkpoints"
os.environ["HF_REPO_ID"] = "Sameer0313/kashmiri-whisper-ultra-lora"

!python3 /kaggle/working/cipher/scripts/finetune-whisper-ultra-accuracy.py

In [ ]:
# 4. Inspect Download Archive
import os
zip_file = "/kaggle/working/kashmiri-whisper-ultra-adapter.zip"
if os.path.exists(zip_file):
    print(f"[+] Downloadable Adapter Ready: {zip_file} ({os.path.getsize(zip_file) / (1024*1024):.2f} MB)")